In [1]:
import librosa
import librosa.display
import numpy as np
import pandas as pd
import os
from typing import List, Tuple, Dict
import matplotlib.pyplot as plt
import random
import cv2
# from vgg16 import VGG16_LSTM
import pickle
import torch.nn as nn
import torch
import datetime
import torch.nn.functional as F
import torchvision.models as models
from torchvision import transforms, utils, models
from torch.utils.data import Dataset, DataLoader
from facenet_pytorch import MTCNN
import logging
import torch
import torch.nn as nn
from vivit_probere1 import ViViTBackbone, ViViTLinearProbe
import gc
from einops import rearrange
from tqdm import tqdm
# from kw_r2plus1d_video_swin_transformer import SwinTransformer3D
batchsz = 4
frame_nums=15
start_epoch = 0
test_flage=False
model_name ="fullshot"
logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)
formatter = logging.Formatter('[%(asctime)s]::%(module)s::%(levelname)s::%(message)s')
streamHandler = logging.StreamHandler()
streamHandler.setFormatter(formatter)
fileHandler = logging.FileHandler('./LOG/personalityLog.log')
fileHandler.setFormatter(formatter)
logger.addHandler(streamHandler)
logger.addHandler(fileHandler)
USE_CUDA = torch.cuda.is_available()
# device = torch.device('cuda:1' if USE_CUDA else 'cpu')

In [2]:
logger.debug('============={}+Video Vision Transformer Strat!============='.format(model_name))
USE_CUDA = torch.cuda.is_available()
print(USE_CUDA)

device = torch.device('cuda:0' if USE_CUDA else 'cpu')
print('학습을 진행하는 기기:', device)

[2026-09-25 01:38:54,066]::3184030073::DEBUG::=============fullshot+Video Vision Transformer Strat!=============


True
학습을 진행하는 기기: cuda:0


In [3]:
def getdata(pathname,datatype):
    if datatype=="train":
        final_data_set=[]
        output_file=open(pathname, "rb")
        for i in range(1):
            # train_data_set.extend(pickle.load(output_file))
            final_data_set.extend(pickle.load(output_file))
            # print(i)
        return final_data_set
    elif datatype=="valid":
        final_data_set=[]
        output_file=open(pathname, "rb")
        for i in range(1):
            valid_data_set=[]
            final_data_set.extend(pickle.load(output_file))
            del valid_data_set
        return final_data_set
print(1)
def reshape_to_expected_input(dataset: List[Tuple[ np.ndarray, np.ndarray]]) -> Tuple[
     np.ndarray, np.ndarray]:
    x0_list = []
    x1_list = []
    for i in range(0, len(dataset)):
        x0_list.append(dataset[i][0])
        x1_list.append(dataset[i][1])
    return (np.stack(x0_list), np.stack(x1_list))

1


In [4]:

save_model_file_path = './save_swintransformer_folder/{}_{}.{}'
mtcnn= MTCNN(image_size=224, margin=0, min_face_size= 20, thresholds=[0.6, 0.7, 0.7], post_process=True)
print(1)

1


In [5]:
train_set_data=getdata("/data1/qx/mean_based_arg/udiva/multimodal_fusion/data/speaking_only/train_set.dat","train")
print(1)
validation_set_data=getdata("/data1/qx/mean_based_arg/udiva/multimodal_fusion/data/speaking_only/valid_set.dat","valid")
print(1)

1
1


In [6]:
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
])
print(1)

1


In [7]:
class ChalearnDataset(Dataset):
    def __init__(self,fullshot,tagdata,transform=None):
        self.fullshot=fullshot
        self.tagdata=tagdata
        self.transform = transform  # 표준화 여부
    def __len__(self):
        return len(self.fullshot)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()#텐서의 경우 목록으로 돌아가기
        fullshot=self.fullshot[idx]
        fullshot=torch.FloatTensor(fullshot)
        big_five_sorces=self.tagdata[idx]
        big_five_sorces = torch.FloatTensor(big_five_sorces)
        return fullshot,big_five_sorces
print(1)

1


In [8]:
num_workerssz = 20

# Linear-probe training settings
probe_lr = 1e-3
probe_weight_decay = 1e-4
probe_epochs = 50
patience = 10
PROBE_SEED = 42

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

print(1)


1


In [9]:
train_set_data = ChalearnDataset(
    fullshot=train_set_data[1],
    tagdata=train_set_data[2],
    transform=transform
)

val_set_data = ChalearnDataset(
    fullshot=validation_set_data[1],
    tagdata=validation_set_data[2],
    transform=transform
)

print(1)


1


In [10]:
def extract_ocean(data,batch_size):
    o=[]
    c=[]
    e=[]
    a=[]
    n=[]
    if batch_size >1:
        for i in range(batch_size):
            o.append(data[i][0].item())
            c.append(data[i][1].item())
            e.append(data[i][2].item())
            a.append(data[i][3].item())
            n.append(data[i][4].item())
        o = torch.tensor(o) 
        c = torch.tensor(c) 
        e = torch.tensor(e) 
        a = torch.tensor(a) 
        n = torch.tensor(n) 
    elif batch_size == 1:
        data=data.squeeze()
        for i in range(batch_size):
            o.append(data[0].item())
            c.append(data[1].item())
            e.append(data[2].item())
            a.append(data[3].item())
            n.append(data[4].item())
            
        o = torch.tensor(o) 
        c = torch.tensor(c) 
        e = torch.tensor(e) 
        a = torch.tensor(a) 
        n = torch.tensor(n)
    return o,c,e,a,n

In [11]:
# ============================================================
# ViViT Video-Only Block-wise Linear Probe -- UDIVA / MSE
# ============================================================
#
# 1. The trained ViViT video-only backbone is loaded and frozen.
# 2. Only the independent probe heads are optimized.
# 3. Evaluation uses MSE directly (lower is better).
# 4. O/C/E/A/N are also reported directly as MSE.
# ============================================================

BACKBONE_CHECKPOINT = "./save_swintransformer_folder/model_best_multiseed.pth"

set_seed(PROBE_SEED)

generator = torch.Generator()
generator.manual_seed(PROBE_SEED)

train_dataloader = DataLoader(
    dataset=train_set_data,
    batch_size=batchsz,
    shuffle=True,
    num_workers=num_workerssz,
    worker_init_fn=seed_worker,
    generator=generator
)

val_dataloader = DataLoader(
    dataset=val_set_data,
    batch_size=batchsz,
    shuffle=False,
    num_workers=num_workerssz,
    worker_init_fn=seed_worker
)

# Build exactly the same ViViT video-only backbone.
backbone = ViViTBackbone(
    2, 14, 14, 2, 2, 2, 5, 96, 6, 10, 8, model=3
).to(device)

checkpoint_data = torch.load(BACKBONE_CHECKPOINT, map_location=device)
state_dict = (
    checkpoint_data["model"]
    if isinstance(checkpoint_data, dict) and "model" in checkpoint_data
    else checkpoint_data
)
backbone.load_state_dict(state_dict, strict=True)

# Frozen backbone + independent linear probes.
model = ViViTLinearProbe(
    backbone=backbone,
    dim=96,
    num_outputs=5
).to(device)

backbone_trainable = sum(
    p.numel() for p in model.backbone.parameters() if p.requires_grad
)
probe_trainable = sum(
    p.numel() for p in model.probe_heads.parameters() if p.requires_grad
)

print(f"Backbone trainable parameters: {backbone_trainable}")
print(f"Probe trainable parameters:    {probe_trainable}")
print(f"Metric: MSE | Patience: {patience}")
assert backbone_trainable == 0, "Backbone must remain frozen."

criterion = nn.MSELoss().to(device)

optimizer = torch.optim.AdamW(
    model.probe_heads.parameters(),
    lr=probe_lr,
    weight_decay=probe_weight_decay
)

probe_names = list(model.probe_heads.keys())
best_mse = {name: float("inf") for name in probe_names}
best_epoch = {name: -1 for name in probe_names}
best_mean_val_mse = float("inf")
no_improve_count = 0

for epoch in range(probe_epochs):
    # -------------------- Train probes only --------------------
    model.train()
    train_sq_sum = {name: 0.0 for name in probe_names}
    train_count = 0

    for fullshot, big_five_data in tqdm(
        train_dataloader,
        desc=f"Probe Train {epoch + 1}/{probe_epochs}"
    ):
        fullshot = fullshot.permute(0, 4, 1, 2, 3).to(device)
        big_five_data = big_five_data.permute(0, 2, 1).squeeze().to(device)
        if big_five_data.ndim == 1:
            big_five_data = big_five_data.unsqueeze(0)

        optimizer.zero_grad()
        outputs = model(fullshot)

        losses = {
            name: criterion(outputs[name], big_five_data)
            for name in probe_names
        }

        total_loss = sum(losses.values())
        total_loss.backward()
        optimizer.step()

        train_count += big_five_data.numel()
        for name in probe_names:
            train_sq_sum[name] += torch.square(
                outputs[name].detach() - big_five_data
            ).sum().item()

    train_mse = {
        name: train_sq_sum[name] / train_count
        for name in probe_names
    }

    # -------------------- Validation --------------------
    model.eval()
    val_sq_sum = {name: 0.0 for name in probe_names}
    val_trait_sq_sum = {
        name: np.zeros(5, dtype=np.float64)
        for name in probe_names
    }
    val_count = 0
    val_sample_count = 0

    with torch.no_grad():
        for fullshot, big_five_data in tqdm(
            val_dataloader,
            desc=f"Probe Valid {epoch + 1}/{probe_epochs}"
        ):
            fullshot = fullshot.permute(0, 4, 1, 2, 3).to(device)
            big_five_data = big_five_data.permute(0, 2, 1).squeeze().to(device)
            if big_five_data.ndim == 1:
                big_five_data = big_five_data.unsqueeze(0)

            outputs = model(fullshot)

            val_count += big_five_data.numel()
            val_sample_count += big_five_data.shape[0]

            for name in probe_names:
                sq_err = torch.square(outputs[name] - big_five_data)
                val_sq_sum[name] += sq_err.sum().item()
                val_trait_sq_sum[name] += (
                    sq_err.sum(dim=0).detach().cpu().numpy()
                )

    val_mse = {
        name: val_sq_sum[name] / val_count
        for name in probe_names
    }
    val_trait_mse = {
        name: val_trait_sq_sum[name] / val_sample_count
        for name in probe_names
    }

    # Each block independently tracks its lowest validation MSE.
    for name in probe_names:
        if val_mse[name] < best_mse[name]:
            best_mse[name] = val_mse[name]
            best_epoch[name] = epoch + 1

    # Early stopping: mean validation MSE across all probe heads.
    mean_val_mse = float(np.mean([val_mse[name] for name in probe_names]))

    if mean_val_mse < best_mean_val_mse:
        best_mean_val_mse = mean_val_mse
        no_improve_count = 0
    else:
        no_improve_count += 1

    print("\n" + "-" * 132)
    print(
        f"Epoch {epoch + 1:02d} | "
        f"Mean Val MSE={mean_val_mse:.6f} | "
        f"Best Mean Val MSE={best_mean_val_mse:.6f} | "
        f"Patience={no_improve_count}/{patience}"
    )

    for name in probe_names:
        o_mse, c_mse, e_mse, a_mse, n_mse = val_trait_mse[name]
        print(
            f"{name:<10} | "
            f"Train MSE {train_mse[name]:.6f} | "
            f"Val MSE {val_mse[name]:.6f} | "
            f"Best MSE {best_mse[name]:.6f} | "
            f"O MSE {o_mse:.6f} | C MSE {c_mse:.6f} | "
            f"E MSE {e_mse:.6f} | A MSE {a_mse:.6f} | "
            f"N MSE {n_mse:.6f}"
        )

    if no_improve_count >= patience:
        print(
            f"\nEarly stopping at epoch {epoch + 1} | "
            f"Best Mean Val MSE={best_mean_val_mse:.6f} | "
            f"Patience={no_improve_count}/{patience}"
        )
        break

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ============================================================
# Final Probe Summary -- MSE only
# ============================================================
print("\n" + "=" * 72)
print("FINAL ViViT VIDEO-ONLY LINEAR PROBE RESULTS -- UDIVA / MSE")
print("=" * 72)
print(
    f"{'Representation':<18}"
    f"{'Best Epoch':>12}"
    f"{'Best MSE':>16}"
)
print("-" * 72)

for name in probe_names:
    print(
        f"{name:<18}"
        f"{best_epoch[name]:>12d}"
        f"{best_mse[name]:>16.6f}"
    )

print("=" * 72)

del model
del backbone
del optimizer
del train_dataloader
del val_dataloader
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()


Backbone trainable parameters: 0
Probe trainable parameters:    4062
Metric: MSE | Patience: 10


Probe Valid 1/50: 100%|██████████| 191/191 [00:17<00:00, 10.87it/s]



------------------------------------------------------------------------------------------------------------------------------------
Epoch 01 | Mean Val MSE=1.348711 | Best Mean Val MSE=1.348711 | Patience=0/10
block1     | Train MSE 0.053008 | Val MSE 1.383717 | Best MSE 1.383717 | O MSE 1.053312 | C MSE 1.389499 | E MSE 1.273934 | A MSE 1.745283 | N MSE 1.456558
block2     | Train MSE 0.047761 | Val MSE 1.370297 | Best MSE 1.370297 | O MSE 1.038098 | C MSE 1.281097 | E MSE 1.247627 | A MSE 1.767867 | N MSE 1.516796
block3     | Train MSE 0.046417 | Val MSE 1.376453 | Best MSE 1.376453 | O MSE 1.022483 | C MSE 1.346962 | E MSE 1.246832 | A MSE 1.770273 | N MSE 1.495717
block4     | Train MSE 0.057134 | Val MSE 1.333140 | Best MSE 1.333140 | O MSE 1.014109 | C MSE 1.268530 | E MSE 1.181206 | A MSE 1.730972 | N MSE 1.470884
block5     | Train MSE 0.048459 | Val MSE 1.340821 | Best MSE 1.340821 | O MSE 1.023299 | C MSE 1.309477 | E MSE 1.233271 | A MSE 1.692993 | N MSE 1.445063
block6  

Probe Valid 2/50: 100%|██████████| 191/191 [00:17<00:00, 11.04it/s]



------------------------------------------------------------------------------------------------------------------------------------
Epoch 02 | Mean Val MSE=1.334514 | Best Mean Val MSE=1.334514 | Patience=0/10
block1     | Train MSE 0.008135 | Val MSE 1.360277 | Best MSE 1.360277 | O MSE 1.061334 | C MSE 1.364985 | E MSE 1.225534 | A MSE 1.751188 | N MSE 1.398344
block2     | Train MSE 0.007232 | Val MSE 1.373210 | Best MSE 1.370297 | O MSE 1.065620 | C MSE 1.349876 | E MSE 1.226865 | A MSE 1.775648 | N MSE 1.448040
block3     | Train MSE 0.006578 | Val MSE 1.360010 | Best MSE 1.360010 | O MSE 1.039973 | C MSE 1.344976 | E MSE 1.204395 | A MSE 1.761678 | N MSE 1.449029
block4     | Train MSE 0.005854 | Val MSE 1.308426 | Best MSE 1.308426 | O MSE 1.029060 | C MSE 1.279779 | E MSE 1.141613 | A MSE 1.724159 | N MSE 1.367517
block5     | Train MSE 0.005359 | Val MSE 1.321137 | Best MSE 1.321137 | O MSE 1.028695 | C MSE 1.310004 | E MSE 1.194200 | A MSE 1.697446 | N MSE 1.375341
block6  

...


------------------------------------------------------------------------------------------------------------------------------------
Epoch 33 | Mean Val MSE=1.314309 | Best Mean Val MSE=1.297492 | Patience=8/10
block1     | Train MSE 0.003935 | Val MSE 1.331798 | Best MSE 1.318241 | O MSE 1.013086 | C MSE 1.285245 | E MSE 1.190248 | A MSE 1.743931 | N MSE 1.426480
block2     | Train MSE 0.003624 | Val MSE 1.325916 | Best MSE 1.313103 | O MSE 1.024505 | C MSE 1.276394 | E MSE 1.179282 | A MSE 1.740205 | N MSE 1.409192
block3     | Train MSE 0.003113 | Val MSE 1.322246 | Best MSE 1.306099 | O MSE 1.036223 | C MSE 1.264285 | E MSE 1.173630 | A MSE 1.735359 | N MSE 1.401732
block4     | Train MSE 0.002811 | Val MSE 1.311494 | Best MSE 1.293649 | O MSE 1.020926 | C MSE 1.276359 | E MSE 1.156701 | A MSE 1.723073 | N MSE 1.380410
block5     | Train MSE 0.002649 | Val MSE 1.306157 | Best MSE 1.287644 | O MSE 1.022285 | C MSE 1.271962 | E MSE 1.164623 | A MSE 1.710450 | N MSE 1.361464
block6  

Probe Valid 34/50: 100%|██████████| 191/191 [00:25<00:00,  7.57it/s]



------------------------------------------------------------------------------------------------------------------------------------
Epoch 34 | Mean Val MSE=1.315540 | Best Mean Val MSE=1.297492 | Patience=9/10
block1     | Train MSE 0.003885 | Val MSE 1.329889 | Best MSE 1.318241 | O MSE 1.011674 | C MSE 1.301910 | E MSE 1.221047 | A MSE 1.735603 | N MSE 1.379211
block2     | Train MSE 0.003556 | Val MSE 1.327582 | Best MSE 1.313103 | O MSE 1.023183 | C MSE 1.293235 | E MSE 1.215671 | A MSE 1.731106 | N MSE 1.374713
block3     | Train MSE 0.003094 | Val MSE 1.326523 | Best MSE 1.306099 | O MSE 1.033346 | C MSE 1.290010 | E MSE 1.215774 | A MSE 1.730799 | N MSE 1.362688
block4     | Train MSE 0.002804 | Val MSE 1.312370 | Best MSE 1.293649 | O MSE 1.016678 | C MSE 1.292389 | E MSE 1.189561 | A MSE 1.719810 | N MSE 1.343411
block5     | Train MSE 0.002639 | Val MSE 1.307578 | Best MSE 1.287644 | O MSE 1.018201 | C MSE 1.284909 | E MSE 1.195810 | A MSE 1.706670 | N MSE 1.332297
block6  

Probe Valid 35/50: 100%|██████████| 191/191 [00:28<00:00,  6.79it/s]


------------------------------------------------------------------------------------------------------------------------------------
Epoch 35 | Mean Val MSE=1.313146 | Best Mean Val MSE=1.297492 | Patience=10/10
block1     | Train MSE 0.003865 | Val MSE 1.329528 | Best MSE 1.318241 | O MSE 1.005019 | C MSE 1.306583 | E MSE 1.207592 | A MSE 1.729059 | N MSE 1.399385
block2     | Train MSE 0.003541 | Val MSE 1.327867 | Best MSE 1.313103 | O MSE 1.019171 | C MSE 1.300544 | E MSE 1.207004 | A MSE 1.727468 | N MSE 1.385149
block3     | Train MSE 0.003060 | Val MSE 1.324036 | Best MSE 1.306099 | O MSE 1.027777 | C MSE 1.290360 | E MSE 1.201576 | A MSE 1.727532 | N MSE 1.372934
block4     | Train MSE 0.002762 | Val MSE 1.309739 | Best MSE 1.293649 | O MSE 1.012506 | C MSE 1.285649 | E MSE 1.180552 | A MSE 1.715410 | N MSE 1.354579
block5     | Train MSE 0.002607 | Val MSE 1.304371 | Best MSE 1.287644 | O MSE 1.013254 | C MSE 1.279797 | E MSE 1.190063 | A MSE 1.702878 | N MSE 1.335862
block6 